In [ ]:
# ! pip install pandas matplotlib spacy gensim wordcloud pyLDAvis
# !python -m spacy download nl_core_news_sm

In [ ]:
# Basisbibliotheken voor data manipulatie en datumtijd
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter

# Visualisatiebibliotheken
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

# NLP en Machine Learning bibliotheken
import spacy
import gensim
from gensim import corpora
from gensim.models import Word2Vec

# Parameters en bestandsnamen instellen
afdelingsnaam = 'avondster'
filename_profielen = f'../zorgdata/gci_clienten_{afdelingsnaam}.csv'
filename_rapportages = f'../zorgdata/gci_rapportages_{afdelingsnaam}.csv'
num_topics = 5
seed = 6
no_below = 2   # Minimaal aantal rapportages waarin het woord moet voorkomen
no_above = 0.5 # Woorden die in meer van de documenten voorkomen worden gefilterd

# Laden van het Nederlands model voor SpaCy
# Het uitschakelen van onnodige componenten versnelt de verwerkingstijd
nlp = spacy.load('nl_core_news_sm', disable=['parser', 'tagger', 'ner'])

In [ ]:
# Functie om de dag van de week en het weeknummer om te zetten naar een datum
def weekdag_naar_datum(weeknummer, dag_van_de_week, jaar=2024):
    dag_index = ['maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag'].index(dag_van_de_week) + 1
    return datetime.fromisocalendar(jaar, weeknummer, dag_index).strftime('%Y-%m-%d')

In [ ]:
# Functie om de DataFrame te transformeren
def transformeer_rapportages(df_rapportages):
    # Bereken de datum en voeg de 'onrust' kolom toe
    df_rapportages['datum'] = df_rapportages.apply(lambda row: weekdag_naar_datum(row['weekno'], row['dag'], jaar=2024), axis=1)
    # Wijzig de onrustscore in een boolean: Wel/geen onrust
    df_rapportages['onrust'] = df_rapportages['onrustscore'] > 50

    # Hernoem kolommen en selecteer relevante kolommen
    df_rapportages.rename(columns={'niveau': 'discipline', 'client_id': 'ct_id'}, inplace=True)
    return df_rapportages[['ct_id', 'datum', 'discipline', 'rapportage', 'onrust']]

In [ ]:
# Lees data in
df_rapportages = pd.read_csv(filename_rapportages, index_col=False)
# Pas de transformatie functie toe
df = transformeer_rapportages(df_rapportages).copy()

In [ ]:
def preprocess_text(text, nlp_model):
    doc = nlp_model(text)
    # Lowercasing, tokenisatie, lemmatisering, stopwoordverwijdering en woordselectie obv part-of-speech
    cleaned_tokens = [token.lemma_.lower() for token in doc if token.is_alpha and not token.is_stop and token.pos_ in ['VERB', 'NOUN', 'ADJ', 'ADV', 'INTJ']]
    # Terug omzetten naar een string
    return " ".join(cleaned_tokens)

# Pas de functie toe op de dataframe
df['rapportage_clean'] = df['rapportage'].apply(lambda x: preprocess_text(x, nlp))

In [ ]:
# Functie om de meest voorkomende woorden te tellen
def count_most_common_words(column, num_words=10):
    all_words = ' '.join(column).split()
    word_counts = Counter(all_words)
    return word_counts.most_common(num_words)

# Meest voorkomende woorden in de originele rapportages
most_common_orig = count_most_common_words(df['rapportage'])
print("Meest voorkomende woorden in originele rapportages:")
print(most_common_orig)

# Meest voorkomende woorden in de schone (preprocessed) rapportages
most_common_clean = count_most_common_words(df['rapportage_clean'])
print("\nMeest voorkomende woorden in schone rapportages:")
print(most_common_clean)

In [ ]:
# Bereken lengtes in karakters en woorden
df['rapportage_len_chars'] = df['rapportage'].apply(len)
df['rapportage_clean_len_chars'] = df['rapportage_clean'].apply(len)
df['rapportage_len_words'] = df['rapportage'].apply(lambda x: len(x.split()))
df['rapportage_clean_len_words'] = df['rapportage_clean'].apply(lambda x: len(x.split()))

def plot_histogram(data, title, color, subplot_index, x_label, y_label='Frequentie', bins=30):
    plt.subplot(1, 4, subplot_index)
    plt.hist(data, bins=bins, color=color, alpha=0.7)
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)

# Instellen van de plotomgeving
plt.figure(figsize=(20, 4))  # grotere breedte om alle plots naast elkaar te passen

# Plotten van de histogrammen
plot_histogram(df['rapportage_len_chars'], 'Lengte in Karakters (Origineel)', 'blue', 1, 'Karakters')
plot_histogram(df['rapportage_clean_len_chars'], 'Lengte in Karakters (Gecleand)', 'green', 2, 'Karakters')
plot_histogram(df['rapportage_len_words'], 'Lengte in Woorden (Origineel)', 'blue', 3, 'Woorden')
plot_histogram(df['rapportage_clean_len_words'], 'Lengte in Woorden (Gecleand)', 'green', 4, 'Woorden')

plt.tight_layout()
plt.show()


In [ ]:
# Functie om wordclouds te genereren voor de topics die we gaan zoeken
def plot_wordclouds(lda_model, dictionary):
    plt.figure(figsize=(20, 10))  # Aanpassen van de breedte en hoogte van de plot
    for idx in range(lda_model.num_topics):
        plt.subplot(1, lda_model.num_topics, idx + 1)
        topic_words = dict(lda_model.show_topic(idx, 30))
        cloud = WordCloud(background_color='white').generate_from_frequencies(topic_words)
        plt.imshow(cloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Topic ' + str(idx+1))

    plt.show()

In [ ]:
# Topic modelling

# Tokeniseren van de gecleande rapportages met SpaCy
tokenized_docs = [[token.text for token in nlp(doc)] for doc in df['rapportage_clean']]

# Creëren van een dictionary
dictionary = corpora.Dictionary(tokenized_docs)
# Filter woorden die in meer dan 50% van de documenten voorkomen of in minder dan 2 documenten
dictionary.filter_extremes(no_below=no_below, no_above=no_above)

# Omzetten van documenten naar een bag-of-words representatie
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

# LDA model training
lda_model = gensim.models.LdaMulticore(corpus, num_topics=num_topics, id2word=dictionary, passes=10, workers=2, random_state=seed)

# Plot de topics
plot_wordclouds(lda_model, dictionary)

In [ ]:
# Visualiseer met pyLDAvis 
lda_display = gensimvis.prepare(lda_model, corpus, dictionary, sort_topics=False)
pyLDAvis.display(lda_display)

In [ ]:
# Functie om topic verdelingen te berekenen
def get_topic_distributions(lda_model, corpus):
    topic_distributions = []

    for doc_bow in corpus:
        doc_topics = lda_model.get_document_topics(doc_bow, minimum_probability=0)
        topic_distribution = {f'topic_{i}': 0 for i in range(num_topics)}
        for topic, prob in doc_topics:
            topic_distribution[f'topic_{topic}'] = prob
        topic_distributions.append(topic_distribution)
    
    return topic_distributions

# Bereken de topic verdelingen voor elk document
topic_dists = get_topic_distributions(lda_model, corpus)

# Voeg de topic verdelingen toe aan de DataFrame
for topic in range(num_topics):
    df[f'topic_{topic}'] = [dist[f'topic_{topic}'] for dist in topic_dists]


In [ ]:
# Word embeddings
# Voorbereiden van de tekstdata
texts = [doc.split() for doc in df['rapportage_clean']]  # Zorg ervoor dat uw data is gesplitst in tokens
# Train het Word2Vec-model
word2vec_model = Word2Vec(sentences=texts, vector_size=50, window=5, min_count=1, workers=4)

In [ ]:
# Gebruik de getrainde model
word_embedding = word2vec_model.wv['onrust']  
print(word_embedding)

In [ ]:
# Vind de woorden die het meest vergelijkbaar zijn met 'onrust'
similar_words = word2vec_model.wv.most_similar('onrust', topn=10)

# Print de resultaten
for word, similarity in similar_words:
    print(f"Word: {word}, Similarity: {similarity}")


In [ ]:
def calculate_document_embedding(text, model):
    embeddings = []
    for word in text.split():
        if word in model.wv:
            embeddings.append(model.wv[word])
    
    # Als het document geen woorden bevat die in het model aanwezig zijn, geef een nulvector terug
    if not embeddings:
        return pd.Series(np.zeros(model.vector_size))
    
    # Bereken het gemiddelde van alle embeddings
    mean_embedding = np.mean(embeddings, axis=0)
    return pd.Series(mean_embedding)

# Bereken de document embeddings voor elke 'rapportage_clean' en voeg deze toe aan de DataFrame
embedding_columns = [f'embedding_{i}' for i in range(word2vec_model.vector_size)]
df[embedding_columns] = df['rapportage_clean'].apply(lambda x: calculate_document_embedding(x, word2vec_model))